# Optional Lab: Redis verstehen und eine Job-Queue bauen

**Vom Key-Value-Store zur Erweiterung unseres Docker-Compose-Labs**

Dieses Notebook ist zugleich Einführung, kommentiertes Skript für eine Live-Demo
und Arbeitsauftrag. Alle Beispiele laufen ohne OpenAI-/Stability-Key: Eine langsame
Python-Funktion simuliert die Arbeit, für die unser bestehendes Lab Bilder generiert.

**Lernziele:** Du kannst erklären, was Redis speichert und wie Clients darauf zugreifen;
Strings, Hashes, TTLs und Listen ausprobieren; Producer, Queue und Worker unterscheiden;
einen RQ-Job ohne Webserver und anschließend über FastAPI ausführen; schließlich
Redis und Worker in dein bestehendes Compose-Lab integrieren.

**Zeitplan:** Grundlagen und direkte Redis-Demos ca. 25 Minuten; Queues/RQ ca. 25 Minuten;
FastAPI ca. 20 Minuten; eigenständige Compose-Erweiterung ca. 45–60 Minuten.
Vorausgesetzt werden Python-Grundlagen, HTTP/FastAPI und das bisherige Docker-Lab.

> **Für die Lehrperson:** Vor jeder Demo stehen eine Vorhersagefrage und die erwartete
> Beobachtung. Die Codezellen können nach dem Setup von oben nach unten ausgeführt werden.
> Worker und Demo-API starten in eigenen Prozessen; die letzte Codezelle beendet sie.


## 1 · Was ist Redis?

Redis ist ein **separater Server für Datenstrukturen**. Ein Client – etwa unser Python-Programm –
sendet Befehle über eine Netzwerkverbindung. Der Server verwaltet Schlüssel und Werte
und antwortet mit einem Ergebnis. `redis-py` ist die Python-Bibliothek dafür, **nicht der Server**.

Der aktive Datenbestand liegt überwiegend im Arbeitsspeicher (RAM). Dadurch sind viele
Operationen sehr schnell. Redis kann mehr als einen einfachen Textwert ablegen: Es unterstützt
unter anderem Hashes, Listen, Sets und sortierte Mengen. Typische Anwendungen sind Caches,
Sitzungsdaten, Zähler, Ranglisten und die Speicherung von Aufgaben für Worker.

![Redis: Clients, Schlüssel und Datenstrukturen im RAM sowie optionale Persistenz](figures/redis-basics.svg)

| Begriff | Bedeutung | Beispiel |
|---|---|---|
| Key | Eindeutiger Name innerhalb einer Redis-Datenbank | `lab:visits` |
| String | Bytefolge; auch Zahlen lassen sich so speichern | `SET lab:name Ada` |
| Hash | Mehrere Felder unter einem Key | Name und Kurs einer Person |
| List | Geordnete Folge; Einfügen/Entnehmen an beiden Enden | Warteschlange |
| TTL | Verbleibende Lebensdauer eines Keys in Sekunden | Cache läuft nach 30 Sekunden ab |

Die Daten werden übers Netzwerk als Bytes übertragen. In den ersten Beispielen lässt
`decode_responses=True` den Client Text daraus machen. Für RQ verwenden wir später bewusst
einen **zweiten Client ohne Text-Decodierung**, da RQ auch binäre Jobdaten speichert.


### Wie funktionieren Befehle, Ablaufzeiten und Persistenz?

1. Der Client sendet beispielsweise `INCR lab:visits`.
2. Redis bearbeitet den Befehl auf seiner Datenstruktur und antwortet mit dem neuen Wert.
3. Mehrere Clients können denselben Key verwenden. `INCR` ist atomar: Gleichzeitige
   Inkremente gehen nicht durch ein konkurrierendes Lesen und Zurückschreiben verloren.
   Eine selbst geschriebene Folge `GET → +1 → SET` ist dagegen **nicht automatisch atomar**.

Viele Befehle sind günstig; Redis ist trotzdem nicht unbegrenzt schnell. Große Werte,
teure Operationen, Netzwerk und verfügbarer RAM spielen eine Rolle. Die einfache Aussage
„Redis ist single-threaded“ ist als Gesamtbeschreibung zu kurz: Für unsere Beispiele ist
entscheidend, dass einzelne Befehle atomar verarbeitet werden, nicht dass jede interne
Arbeit des Servers nur einen Thread hätte.

**RAM heißt nicht zwingend flüchtig:** Mit RDB werden Snapshots geschrieben, mit AOF
werden Änderungen protokolliert. Wie viel bei einem Ausfall verloren gehen kann, hängt
von der Konfiguration ab. Für Container müssen Persistenzkonfiguration **und** ein Volume
für das Datenverzeichnis zusammenpassen. Ein Volume allein schaltet AOF nicht ein.
Eine TTL ist davon unabhängig: Ein Key darf auch bei aktivierter Persistenz ablaufen.

**Denkfrage:** Was ist der Unterschied zwischen „der Cache-Eintrag ist abgelaufen“ und
„der Redis-Container wurde gelöscht“?


## 2 · Setup für die Live-Demo

Verwende **Python 3.12**, Docker und Linux/macOS bzw. **WSL2 unter Windows**.
Native Windows-Worker sind in dieser RQ-Version wegen Unix-Signalen nicht unterstützt.
Nutze einen Jupyter-fähigen Editor (z. B. VS Code mit
Python-/Jupyter-Erweiterung). Die Demo verwendet RQ 1.15.1 wie das bestehende Backend;
aktuelle RQ-Tutorials können APIs neuerer Versionen zeigen.

Im Terminal, aus dem Repository-Verzeichnis:

```bash
cd optional-lab-redis
python3.12 -m venv .venv
source .venv/bin/activate       # Unter Windows diese Befehle in WSL2 ausführen.
python -m pip install -r requirements.txt
python -m ipykernel install --user --name optional-redis --display-name "Python (Redis Lab)"
```

Öffne dieses Notebook und wähle den Kernel **Python (Redis Lab)**.
Falls du JupyterLab statt VS Code nutzt: zusätzlich `pip install jupyterlab`, dann `jupyter lab`.

Starte in einem **zweiten Terminal** einen eigenen Redis-Container:

```bash
docker run --rm --name optional-redis   -p 127.0.0.1:16379:6379   redis:7.4-alpine
```

Dieses Terminal bleibt belegt. Redis lauscht im Container auf 6379; der Notebook-Kernel
auf dem Host erreicht es über **127.0.0.1:16379**. Die Bindung an 127.0.0.1 hält die
Demo lokal. Sie nutzt keine produktive Redis-Datenbank und keine Cloud-API-Keys.
Wenn der Container bereits läuft, starte keinen zweiten mit demselben Namen.

Bei einem Kernel **innerhalb eines Containers** zeigt `localhost` auf diesen Container:
Passe `REDIS_URL` auf einen von dort erreichbaren Host bzw. den Redis-Servicenamen an.
Dieses Notebook nimmt zunächst an, dass Kernel und Docker-Port auf demselben Host liegen.

**Noch keine Compose-Lösung:** Der einzelne Redis-Container reicht für die Demos.
Das Zusammenschalten der Dienste in Compose ist später eure Aufgabe.


In [ ]:
import os
import sys
import time
import json
import uuid
from pathlib import Path

# Das Notebook darf aus dem Repo-Root oder aus optional-lab-redis gestartet werden.
LAB_DIR = Path.cwd()
if (LAB_DIR / "optional-lab-redis" / "redis-introduction.ipynb").exists():
    LAB_DIR = LAB_DIR / "optional-lab-redis"
assert (LAB_DIR / "redis-introduction.ipynb").exists(), "Bitte im Repo oder Notebook-Ordner starten."
os.chdir(LAB_DIR)

# Keine .env-Dateien des Bild-Backends laden: Diese Demo braucht keine API-Keys.
REDIS_URL = os.getenv("REDIS_URL", "redis://127.0.0.1:16379/0")
SESSION = uuid.uuid4().hex[:8]  # Eigene Keys und eigene Queue pro Notebook-Durchlauf.
PREFIX = f"optional-lab:{SESSION}"
QUEUE_NAME = f"optional-lab-{SESSION}"
RUNTIME = LAB_DIR / ".runtime"
RUNTIME.mkdir(exist_ok=True)
print("Python:", sys.version.split()[0], "| Demo-Präfix:", PREFIX)


In [ ]:
import redis

# Textclient für unsere eigenen Strings, Hashes und Listen.
redis_text = redis.Redis.from_url(
    REDIS_URL, decode_responses=True, socket_connect_timeout=3, socket_timeout=5
)
assert redis_text.ping() is True  # PING prüft, ob der Server erreichbar ist.
print("Redis antwortet: PONG")


**Erwartet:** `Redis antwortet: PONG`.
Bei `ConnectionError`: Läuft der Container? Stimmt das Port-Mapping? Läuft der Kernel
wirklich auf demselben Host? `docker ps` und `docker logs optional-redis` helfen.
Bei `ModuleNotFoundError`: Prüfe den ausgewählten Notebook-Kernel.


## 3 · Redis ohne FastAPI: Werte, Zähler, Hashes und TTL

**Vorhersage:** Welchen Python-Typ liefert `GET` für einen Zähler? Warum braucht man
für einen sicheren gemeinsamen Zähler nicht erst `GET`, dann `SET`?


In [ ]:
name_key = f"{PREFIX}:name"
counter_key = f"{PREFIX}:visits"
redis_text.set(name_key, "Ada")
redis_text.set(counter_key, 0)

# INCR ist ein einzelner atomarer Redis-Befehl.
for _ in range(3):
    visits = redis_text.incr(counter_key)

print("Name:", redis_text.get(name_key))
print("INCR-Ergebnis:", visits, type(visits).__name__)
print("GET-Ergebnis:", redis_text.get(counter_key), type(redis_text.get(counter_key)).__name__)
assert visits == 3


**Beobachtung:** `INCR` liefert die Zahl `3`, `GET` liefert bei diesem Textclient den
String `"3"`. Redis-Befehle haben unterschiedliche Antworttypen.


In [ ]:
profile_key = f"{PREFIX}:student"
redis_text.hset(profile_key, mapping={"name": "Ada", "course": "Solution Deployment"})
redis_text.hset(profile_key, "completed_labs", 2)
redis_text.hincrby(profile_key, "completed_labs", 1)
print(redis_text.hgetall(profile_key))  # Ein Hash enthält mehrere Felder unter einem Key.


In [ ]:
ttl_key = f"{PREFIX}:temporary"
redis_text.set(ttl_key, "kurz verfügbar", ex=2)  # ex setzt eine TTL in Sekunden.
print("Wert sofort:", redis_text.get(ttl_key))
print("TTL sofort:", redis_text.ttl(ttl_key))
time.sleep(2.2)  # Für die Live-Demo warten wir bewusst auf den Ablauf.
print("Wert danach:", redis_text.get(ttl_key))
print("TTL danach:", redis_text.ttl(ttl_key))
assert redis_text.get(ttl_key) is None


**Erwartet:** Der temporäre Wert ist verschwunden; `TTL` liefert `-2`.
`-1` bedeutet dagegen: Key vorhanden, aber keine Ablaufzeit.

**Mini-Aufgabe (3 Minuten):** Lege einen eigenen Session-Key mit 10 Sekunden TTL an.
Lies ihn sofort und nach dem Ablauf. Erkläre, weshalb eine TTL für einen Cache sinnvoll
ist, für noch nicht bearbeitete Jobs aber besonders sorgfältig gewählt werden muss.
Verwende für eigene Keys weiterhin `PREFIX`.


## 4 · Was ist eine Queue – und weshalb wollen wir eine?

Eine Queue ist eine **Warteschlange für Arbeit**. Ein Producer legt Aufgaben hinein;
ein Consumer bzw. Worker holt sie heraus und bearbeitet sie. Bei FIFO (*first in, first out*)
wird die zuerst eingereihte Aufgabe zuerst entnommen.

In unserem Bild-Lab kann die Generierung mehrere Sekunden dauern. Wenn sie direkt im
HTTP-Aufruf ausgeführt wird, wartet der Client auf das fertige Bild. Eine Queue erlaubt:

- **Schnelle Annahme:** Die API antwortet mit einer Job-ID, bevor das Bild fertig ist.
- **Entkopplung:** API und Worker sind getrennte Prozesse; Worker können später starten.
- **Pufferung:** Kurzzeitig mehr Anfragen als Worker-Kapazität können warten.
- **Separate Skalierung:** Zusätzliche Worker können mehrere Jobs parallel ausführen.

Dafür brauchen wir einen zusätzlichen Dienst, Statusabfragen und eine Strategie für Fehler,
Wiederholungen und Überlast. Die Queue macht **den einzelnen Job nicht schneller**.
Eine dauerhaft wachsende Queue bedeutet, dass mehr Arbeit ankommt als abgearbeitet wird.

![Producer, Redis-Queue, Worker und Ergebnis; FIFO beschreibt nicht die Abschlussreihenfolge](figures/queue-flow.svg)

**Wichtige Trennung:** Redis speichert Daten über Aufgaben. Ein Python-Worker führt die
Python-Funktion aus. Redis führt unseren Bildgenerator nicht selbst aus.
Mehrere Worker können Jobs unterschiedlich schnell beenden: FIFO-Entnahme garantiert
keine FIFO-Fertigstellung.


### Drei Wege im Vergleich

| Ansatz | Wann erhält der Client das Ergebnis? | Wo läuft die Arbeit? | Grenze |
|---|---|---|---|
| Direkter Funktionsaufruf / HTTP-Aufruf | Nach Ende der Arbeit | Im aufrufenden Prozess / API-Prozess | Langer Request, Timeout möglich |
| FastAPI `BackgroundTasks` | Erst Annahme, Ergebnis später abholen | Weiterhin in der API-Anwendung | Keine externe dauerhafte Job-Queue |
| Redis + RQ | Erst Job-ID, Ergebnis später abholen | Separater Worker | Zusätzliche Infrastruktur und Fehlerbehandlung |

`async`/`await` ersetzt keine Worker-Queue. Es hilft bei kooperativem Warten auf I/O,
lagert aber nicht automatisch CPU-Arbeit in einen anderen Prozess aus.
Für die synchronen Bibliotheken in unserer Demo verwenden wir normale `def`-Routen;
FastAPI führt diese in seinem Threadpool aus.


## 5 · Die kleinste Queue: Redis-Liste, noch ohne RQ und FastAPI

**Vorhersage:** In welcher Reihenfolge werden A, B und C abgearbeitet?
`RPUSH` hängt rechts an; `BLPOP` entnimmt links. `RPUSH` + `BRPOP` wäre hier **nicht FIFO**.


In [ ]:
list_key = f"{PREFIX}:simple-queue"
redis_text.delete(list_key)  # Nur unsere eigene Demo-Liste zurücksetzen.
for label in ["A", "B", "C"]:
    # JSON macht die Nutzdaten für diese einfache Demo lesbar.
    redis_text.rpush(list_key, json.dumps({"task": label}))
print("Wartende Aufträge:", redis_text.llen(list_key))
print("Listeninhalt:", redis_text.lrange(list_key, 0, -1))


In [ ]:
processed = []
for _ in range(3):  # Absichtlich endlich: Run All darf nicht ewig blockieren.
    item = redis_text.blpop(list_key, timeout=2)
    if item is None:
        break
    queue_key, payload = item
    task = json.loads(payload)
    time.sleep(0.2)  # Stellvertretend für aufwendige Arbeit.
    processed.append(task["task"])
    print("Bearbeitet:", task["task"])
assert processed == ["A", "B", "C"]
print("Leere Queue:", redis_text.blpop(list_key, timeout=1))


**Beobachtung:** A, B, C; anschließend `None` nach dem Timeout.
`BLPOP` wartet auf der Verbindung des Consumers, nicht durch Blockieren des gesamten
Redis-Servers. Andere Clients können weiterhin Befehle ausführen.

**Grenze dieser Mini-Queue:** Sobald ein Job entnommen wurde, ist er aus der Liste entfernt.
Stürzt der Consumer danach ab, fehlt eine Bestätigung und der Job kann verloren gehen.
Wir haben noch keine Job-IDs, Zustände, Resultate, Wiederholungen oder Fehlerregistrierung.
Auch eine leere Liste bedeutet nicht, dass alle entnommenen Jobs bereits fertig sind.

Für robustere List-Queues sind zusätzliche Mechanismen nötig; Redis Streams bieten
beispielsweise Consumer Groups und Bestätigungen. Für unser Python-Lab verwenden wir
statt einer selbst gebauten Lösung **RQ (Redis Queue)**.


## 6 · RQ ohne FastAPI: eine echte Python-Aufgabe

RQ ergänzt Redis um Python-Jobs, Status, Ergebnisse, Worker und Fehlerregistrierung.
Ein Job enthält unter anderem den Importpfad der Funktion und ihre Argumente.
Producer und Worker müssen dieselbe Funktion importieren können.

**Notebook-Falle:** Eine direkt im Notebook definierte Funktion lebt meist in `__main__`.
RQ kann sie im Worker nicht zuverlässig importieren. Deshalb schreibt die nächste Zelle
ein kleines Modul in `.runtime/`. Diese generierten Demo-Dateien werden nicht eingecheckt.
Die Zelle kann wiederholt ausgeführt werden; sie überschreibt nur dieses Demo-Modul.

**Vorhersage:** Wird nach `enqueue()` bereits Python-Code im Worker ausgeführt, wenn noch
gar kein Worker läuft?


In [ ]:
%%writefile .runtime/redis_lab_tasks.py
import os
import time


def build_report(name: str, delay: float = 1.0) -> dict:
    # Langsame Arbeit simulieren: kein API-Key und keine externen Kosten nötig.
    time.sleep(delay)
    if name == "FAIL":
        # Absichtlicher Fehler für die spätere Live-Demo.
        raise ValueError("Absichtlicher Demo-Fehler")
    return {
        "name": name,
        "word_count": len(name.split()),
        "worker_pid": os.getpid(),  # Belegt, in welchem Prozess die Arbeit lief.
    }


In [ ]:
import importlib
from rq import Queue
from rq.job import Job

# Der Kernel muss das Modul ebenso finden wie der spätere Worker-Prozess.
if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))
importlib.invalidate_caches()
import redis_lab_tasks
importlib.reload(redis_lab_tasks)

# RQ benötigt Bytes, nicht automatisch decodierte Strings!
redis_jobs = redis.Redis.from_url(REDIS_URL, decode_responses=False)
queue = Queue(QUEUE_NAME, connection=redis_jobs)
tracked_jobs = []

started = time.perf_counter()
job = queue.enqueue(
    redis_lab_tasks.build_report, "Mein erster Report", 1.0,
    job_timeout=30, result_ttl=300, failure_ttl=300,
)
tracked_jobs.append(job)
print("Enqueue dauerte:", round(time.perf_counter() - started, 3), "Sekunden")
print("Status ohne Worker:", job.get_status())
print("Wartende Jobs:", queue.count)
assert job.get_status() == "queued"


**Erwartet:** `queued`. Redis kennt den Job, aber noch niemand arbeitet daran.
`job_timeout` begrenzt die Bearbeitung; `result_ttl` ist die Aufbewahrungsdauer nach
Erfolg, `failure_ttl` nach einem Fehler. Diese Werte sind nicht die Wartezeit in der Queue.

RQ nutzt standardmäßig Pickle zur Serialisierung. Deshalb müssen Producer und Redis
vertrauenswürdig sein; fremde Jobdaten dürfen nicht ungeprüft in diese Queue gelangen.

### Einen separaten Worker starten

Für eine kurze Notebook-Demo unter Linux/macOS bzw. WSL2 verwenden wir **`rq.SimpleWorker`** in einem
separaten Prozess. Er führt Jobs direkt in diesem Prozess aus und hat nicht dieselbe
Prozessisolation bzw. Heartbeat-Eigenschaften wie der normale RQ-Worker. Im bestehenden
Docker-Lab bleibt `python worker.py` mit dem normalen Worker die Zielvariante.
`SpawnWorker` aus neueren RQ-Dokumentationen gehört nicht zu unserer Version 1.15.1.

Die Hilfsfunktionen unten starten nur Prozesse dieses Notebooks, schreiben Logs nach
`.runtime/` und ermöglichen gezieltes Aufräumen. Kein Start mit `&`, kein globales `pkill`.


In [ ]:
import atexit
import subprocess

processes = {}  # name -> (Prozess, Logdatei)


def stop_process(name):
    record = processes.pop(name, None)
    if record is None:
        return
    process, log = record
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=8)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    log.close()


def cleanup_processes():
    for name in list(processes):
        stop_process(name)


atexit.register(cleanup_processes)  # Zusätzlich zur expliziten Cleanup-Zelle am Ende.


def start_process(name, arguments):
    assert name not in processes, f"{name} läuft schon; zuerst stop_process aufrufen."
    env = os.environ.copy()
    env.update(REDIS_URL=REDIS_URL, RQ_QUEUE_NAME=QUEUE_NAME)
    log = (RUNTIME / f"{SESSION}-{name}.log").open("w", encoding="utf-8")
    process = subprocess.Popen(arguments, cwd=RUNTIME, env=env, stdout=log, stderr=log)
    processes[name] = (process, log)
    return process


def start_worker(name):
    # CLI der installierten RQ-Version im selben Python wie der Notebook-Kernel starten.
    return start_process(name, [
        sys.executable, "-c", "from rq.cli import main; main()", "worker", "--url", REDIS_URL,
        "--worker-class", "rq.SimpleWorker", "--name", f"{PREFIX}-{name}", QUEUE_NAME,
    ])


def wait_for_job(job, timeout=20):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        state = job.get_status(refresh=True)
        if state in {"finished", "failed", "stopped", "canceled"}:
            return state
        time.sleep(0.1)
    raise TimeoutError("Job nicht fertig. Worker-Log unter .runtime/ prüfen.")


In [ ]:
worker_process = start_worker("worker-1")
assert wait_for_job(job) == "finished"
print("Ergebnis:", job.return_value())
print("Notebook-PID:", os.getpid(), "| Worker-PID:", worker_process.pid)
assert job.return_value()["worker_pid"] != os.getpid()


**Beobachtung:** Der erste Job wird nach dem Worker-Start abgearbeitet. Die PID zeigt:
Der Code lief in einem anderen Prozess. Die API war dafür noch gar nicht nötig.
Das kleine Python-Ergebnis liegt bei RQ in Redis; bei unserem Bild-Lab werden dagegen
JPEG-Dateien auf dem Dateisystem gespeichert.

**Live-Experiment:** Stoppe den Worker nach einem abgeschlossenen Job. Lege neue Arbeit
hinein. Starte dann wieder einen Worker. Bleibt der Auftrag erhalten, solange Redis läuft?


In [ ]:
stop_process("worker-1")  # Es läuft gerade kein Job; wir brechen keine Aufgabe ab.
paused_job = queue.enqueue(redis_lab_tasks.build_report, "Später bearbeiten", 0.5, result_ttl=300)
tracked_jobs.append(paused_job)
time.sleep(0.3)
print("Worker aus; Status:", paused_job.get_status())
assert paused_job.get_status() == "queued"

start_worker("worker-1")
assert wait_for_job(paused_job) == "finished"
print("Nach Neustart:", paused_job.return_value())


### Fehler sichtbar machen

**Vorhersage:** Ist `enqueue()` erfolgreich, obwohl die Funktion später einen Fehler wirft?
Die Annahme des Jobs ist nicht das Ergebnis seiner Ausführung.


In [ ]:
failed_job = queue.enqueue(redis_lab_tasks.build_report, "FAIL", 0.2, failure_ttl=300)
tracked_jobs.append(failed_job)
assert wait_for_job(failed_job) == "failed"
failed_job.refresh()  # Fehlerdetails aus Redis laden.
print("Status:", failed_job.get_status())
print("Fehler:", failed_job.exc_info.strip().splitlines()[-1])


RQ registriert den Fehler; es wiederholt diesen Job hier **nicht automatisch**.
Retries müssten bewusst konfiguriert werden. Auch RQ garantiert nicht, dass jede
fachliche Aktion genau einmal stattfindet: Bei Wiederholung muss eine Aufgabe möglichst
**idempotent** sein, also z. B. nicht dieselbe Zahlung zweimal auslösen.

**Mini-Aufgabe (5 Minuten):** Starte mit `start_worker("worker-2")` einen zweiten Worker,
enqueue drei Reports und vergleiche deren `worker_pid`. Verwende pro Job dieselben
`result_ttl`/`failure_ttl` wie oben und sammle sie in `tracked_jobs`.
Beende danach `worker-2`. Erkläre, warum ein später gestarteter kurzer Job früher fertig
werden kann als ein langer Job. Höhere Parallelität bedeutet nicht automatisch kürzere
Laufzeit für jeden einzelnen Report.


## 7 · Dasselbe Prinzip mit FastAPI

Jetzt wird FastAPI zum Producer. Wir bieten zwei Wege an:

- `POST /reports/sync`: Report wird während des Requests berechnet, Antwort ist das Ergebnis.
- `POST /reports`: Job wird eingereiht, Antwort ist **202 Accepted** mit einer Job-ID.
- `GET /reports/{job_id}`: Status abfragen; bei `finished` steht das Ergebnis in `result`.

Ein HTTP-Status 202 bedeutet **angenommen**, nicht „fertig“. Ein Job kann später scheitern.
Diese Demo liefert für bekannte Statusabfragen HTTP 200 und einen fachlichen Status im JSON;
eine unbekannte oder abgelaufene Job-ID liefert 404. Das bestehende Bild-Lab hat einen
anderen Vertrag: `POST /v3/images` liefert dort bereits HTTP 200 mit `img_id`, und das
Polling über `GET /images/{img_id}` liefert 202 bzw. JPEG mit HTTP 200.

**Vorhersage:** Welche POST-Variante antwortet schneller? Wird der Report durch die Queue
selbst schneller berechnet?


In [ ]:
%%writefile .runtime/redis_lab_api.py
import os

import redis
from fastapi import FastAPI, HTTPException
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
from rq import Queue
from rq.job import Job
from rq.exceptions import NoSuchJobError

from redis_lab_tasks import build_report

app = FastAPI(title="Redis Queue – Live-Demo")
connection = redis.Redis.from_url(os.environ["REDIS_URL"], decode_responses=False)
queue = Queue(os.environ["RQ_QUEUE_NAME"], connection=connection)


class ReportRequest(BaseModel):
    name: str = Field(min_length=1, max_length=80)
    delay: float = Field(default=1.0, ge=0, le=5)


@app.exception_handler(redis.exceptions.RedisError)
async def redis_unavailable(request, exc):
    # Ein Infrastrukturfehler soll keine erfolgreiche Job-Annahme vortäuschen.
    return JSONResponse(status_code=503, content={"detail": "Redis ist nicht erreichbar."})


@app.post("/reports/sync")
def report_sync(request: ReportRequest):
    # Normales def: Die blockierende Demo-Funktion läuft im FastAPI-Threadpool.
    return build_report(request.name, request.delay)


@app.post("/reports", status_code=202)
def enqueue_report(request: ReportRequest):
    # Die API führt build_report hier NICHT aus. Das macht der Worker später.
    job = queue.enqueue(
        build_report, request.name, request.delay,
        job_timeout=30, result_ttl=300, failure_ttl=300,
    )
    return {"job_id": job.id, "status_url": f"/reports/{job.id}"}


@app.get("/reports/{job_id}")
def report_status(job_id: str):
    try:
        job = Job.fetch(job_id, connection=connection)
    except NoSuchJobError:
        raise HTTPException(status_code=404, detail="Job unbekannt oder bereits abgelaufen.")
    if job.origin != queue.name:
        raise HTTPException(status_code=404, detail="Job gehört nicht zu dieser Demo.")
    state = job.get_status(refresh=True)
    body = {"job_id": job.id, "status": state}
    if state == "finished":
        body["result"] = job.return_value()  # Status prüfen, nicht nur result is None!
    elif state == "failed":
        body["error"] = "Report fehlgeschlagen; Worker-Log prüfen."
    return body


In [ ]:
import socket
import httpx

# Einen freien lokalen Port wählen, damit bestehende Apps nicht gestört werden.
with socket.socket() as sock:
    sock.bind(("127.0.0.1", 0))
    api_port = sock.getsockname()[1]
API_URL = f"http://127.0.0.1:{api_port}"
api_process = start_process("api", [
    sys.executable, "-m", "uvicorn", "redis_lab_api:app",
    "--host", "127.0.0.1", "--port", str(api_port),
])
http = httpx.Client(base_url=API_URL, timeout=10, trust_env=False)

# Auf Bereitschaft warten, nicht einfach annehmen, dass Popen bereits 'bereit' bedeutet.
deadline = time.monotonic() + 15
while time.monotonic() < deadline:
    if api_process.poll() is not None:
        raise RuntimeError("API beendet; .runtime/-Log prüfen.")
    try:
        response = http.get("/openapi.json")
        if response.status_code == 200:
            break
    except httpx.ConnectError:
        pass
    time.sleep(0.1)
else:
    raise TimeoutError("API wurde nicht bereit.")
print("Interaktive API-Dokumentation:", API_URL + "/docs")


In [ ]:
# Vergleich: dieselbe simulierte Arbeit, zwei verschiedene Antwortzeitpunkte.
payload = {"name": "FastAPI Live Report", "delay": 1.5}
started = time.perf_counter()
sync_response = http.post("/reports/sync", json=payload)
sync_response.raise_for_status()
print("Direkte Antwort nach", round(time.perf_counter() - started, 2), "Sekunden:")
print(sync_response.json())

started = time.perf_counter()
queued_response = http.post("/reports", json=payload)
assert queued_response.status_code == 202
print("Job-Annahme nach", round(time.perf_counter() - started, 2), "Sekunden:")
print(queued_response.json())
api_job = Job.fetch(queued_response.json()["job_id"], connection=redis_jobs)
tracked_jobs.append(api_job)


In [ ]:
def poll_report(status_url, timeout=20):
    deadline = time.monotonic() + timeout
    previous = None
    while time.monotonic() < deadline:
        response = http.get(status_url)
        response.raise_for_status()
        body = response.json()
        if body["status"] != previous:
            print("Status:", body["status"])
            previous = body["status"]
        if body["status"] in {"finished", "failed", "stopped", "canceled"}:
            return body
        time.sleep(0.2)  # Polling mit Pause und oberer Zeitgrenze.
    raise TimeoutError("Kein Endzustand: Läuft ein Worker für die richtige Queue?")

report = poll_report(queued_response.json()["status_url"])
assert report["status"] == "finished"
print("Ergebnis:", report["result"])


**Beobachtung:** Die direkte Antwort braucht mindestens die simulierte Arbeitszeit.
Die Queue-Antwort enthält früh eine ID. Das Resultat erscheint trotzdem erst nach der
Berechnung. Abhängig vom Timing kann Polling einzelne Zwischenzustände überspringen.

Die API-Prozess-ID im synchronen Ergebnis unterscheidet sich von der Worker-PID im
Queue-Ergebnis. Während eines fertigen Demo-Laufs bleibt nur der separat gestartete
Worker für die Queue zuständig.

**Live-Alternative im Browser:** Öffne die ausgegebene `/docs`-URL, sende einen Report
mit `delay=3` und frage anschließend die ID ab. In Remote-Entwicklungsumgebungen musst du
den ausgegebenen Port gegebenenfalls zum Browser weiterleiten.


In [ ]:
# Einen Fehler über dieselbe HTTP-Schnittstelle beobachten.
response = http.post("/reports", json={"name": "FAIL", "delay": 0.2})
assert response.status_code == 202  # Annahme klappt, obwohl die Aufgabe später scheitert.
tracked_jobs.append(Job.fetch(response.json()["job_id"], connection=redis_jobs))
failed_report = poll_report(response.json()["status_url"])
assert failed_report["status"] == "failed"
print(failed_report)

assert http.get("/reports/nicht-vorhanden").status_code == 404
assert http.post("/reports", json={"name": "", "delay": 1}).status_code == 422
print("Unbekannter Job: 404; ungültige Eingabe: 422")


### Verständnisfragen vor dem Transfer

1. Warum kann `POST /reports` erfolgreich antworten und der Job später trotzdem scheitern?
2. Was passiert, wenn Redis erreichbar ist, aber kein Worker läuft?
3. Was ist der Unterschied zwischen API-Prozess, Redis-Server und Worker-Prozess?
4. Wo liegen RQ-Ergebnisse dieser Demo? Wo liegen dagegen die Bilder unseres Labs?
5. Wann würdest du bewusst bei einem direkten Request oder `BackgroundTasks` bleiben?
6. Welche zusätzlichen Fragen entstehen bei Wiederholungen, vielen Jobs oder einem Neustart?

**Diskussionshinweis:** Ergebnisse haben eine Lebensdauer. Ein abgelaufener Job ist nicht
dasselbe wie ein fehlgeschlagener Job. Für eine reale Anwendung müsste man außerdem
Zugriffsschutz, Backpressure, Idempotenz und eine passende Persistenzstrategie gestalten.


## 8 · Demo gezielt aufräumen

Die folgende Zelle beendet ausschließlich unsere Worker/API-Prozesse, löscht unsere
aufgezeichneten Jobs und Demo-Keys. **Kein `FLUSHDB` und kein globales Stoppen aller Container.**
Wenn du später eine frühere Demo wiederholen möchtest, starte das Notebook mit einem frischen
Kernel von vorn; dadurch erhältst du eine neue isolierte Queue und ein neues Key-Präfix.


In [ ]:
cleanup_processes()  # Worker auch dann stoppen, wenn der API-Teil noch nicht lief.
if "http" in globals():
    http.close()
for demo_job in tracked_jobs:
    # Jobs können inzwischen durch ihre TTL entfernt worden sein.
    try:
        demo_job.delete()
    except Exception as exc:
        from rq.exceptions import NoSuchJobError
        if not isinstance(exc, NoSuchJobError):
            raise
queue.delete(delete_jobs=True)  # Nur die zufällig benannte Queue dieses Durchlaufs.
redis_text.delete(name_key, counter_key, profile_key, ttl_key, list_key)
redis_jobs.close()
redis_text.close()
print("Demo-Prozesse beendet; eigene Jobs und Keys entfernt.")


Wenn du den Demo-Container nicht mehr brauchst, beende ihn gezielt im Terminal:

```bash
docker stop optional-redis
```

Er wurde mit `--rm` gestartet und wird entfernt. Diese Demo hat absichtlich kein
Persistenz-Volume eingerichtet. Die generierten Python-Dateien und Logs in `.runtime/`
kannst du für die nächste Vorführung behalten; sie sind in Git ignoriert.


## 9 · Eure Aufgabe: Das vorhandene Compose-Lab um Redis erweitern

**Arbeitsauftrag:** Erweitert eure Lösung aus [lab-assignments.ipynb](../lab-assignments.ipynb),
sodass die Bildgenerierung über Redis und einen separaten RQ-Worker läuft. Arbeitet in
[`docker-lab/docker-compose.yml`](../docker-lab/docker-compose.yml).
Die bisherigen Dockerfiles bleiben eure Aufgabe; dieses Notebook liefert keine fertige
Compose-Datei zum Kopieren.

![Zielarchitektur: Frontend, Backend, Redis, Worker und gemeinsames Bild-Volume](figures/compose-architecture.svg)

### Schritt A – Den vorhandenen Code zuordnen

Lest die folgenden Dateien und markiert Producer, Queue-Verbindung, Worker und Ergebnisspeicher:

- [`backend/routes/image_gen_router.py`](../docker-lab/backend/routes/image_gen_router.py):
  `POST /v3/images` reiht `worker.gen_image_task` ein. Die Antwort enthält `img_id` (HTTP 200).
- [`backend/dependencies.py`](../docker-lab/backend/dependencies.py):
  `REDIS_URL` steuert die Verbindung; ohne Queue-Namen verwendet RQ `default`.
- [`backend/worker.py`](../docker-lab/backend/worker.py):
  `python worker.py` hört auf `high`, `default`, `low` und speichert das generierte JPEG.
- `GET /images/{img_id}` liefert 202, solange die Datei nicht existiert, danach das JPEG.
- [`frontend/streamlit_app.py`](../docker-lab/frontend/streamlit_app.py):
  Wählt **v3**, damit ihr tatsächlich den Redis-Pfad benutzt.

**Achtung beim Transfer:** Verwendet die vorhandenen Endpunkte und `img_id`, nicht die
`/reports`-Routen der Demo. Die Bildgenerierung benötigt gültige Provider-Zugänge und Guthaben;
die vorangehenden Redis-/Report-Demos benötigen beides nicht.


### Schritt B – Vier Services planen und konfigurieren

Ergänzt neben `frontend` und `backend` die Services **`redis`** und **`worker`**.
Der Worker darf dasselbe gebaute Image wie das Backend verwenden, erhält aber ein anderes
Startkommando: `python worker.py`. Nutzt eine passende Python-Version, z. B. Python 3.12.

| Ort | Konfigurationswert / Anforderung | Eure Erklärung |
|---|---|---|
| Frontend | `API_URL=http://backend:8000` | Warum nicht `localhost`? |
| Backend **und** Worker | `REDIS_URL=redis://redis:6379/0` | Warum derselbe Server und dieselbe DB? |
| Backend **und** Worker | Gleicher absoluter `UPLOAD_FOLDER` | Wer schreibt, wer liest? |
| Worker | Start mit `python worker.py`; gleiche Task-Module wie im Backend | Warum reicht der Redis-Container nicht? |
| Redis | Port 6379 innerhalb des Compose-Netzwerks | Muss dieser Port auf dem Host veröffentlicht werden? |

In Compose ist der **Servicename der DNS-Name**. `localhost` bezeichnet immer den aktuellen
Container. Kommunikation zwischen Containern benutzt den Container-Port, nicht ein
beliebiges Host-Port-Mapping. Redis muss für diese Anwendung nicht auf dem Host erreichbar sein.
Konfiguriert die benötigten Provider-Variablen zur Laufzeit, etwa über `env_file` oder
`environment`; backt Zugangsdaten nicht in das Image.

**Denkt zuerst nach, schreibt dann YAML:** Welche zwei Prozesse führen tatsächlich
Bildgenerierung aus? Welche benötigen ausschließlich eine URL? Welche Einstellungen müssen
Backend und Worker gemeinsam haben?


### Schritt C – Bereitschaft und zwei unterschiedliche Volumes

1. **Redis-Bereitschaft:** Definiert einen Healthcheck mit `redis-cli ping`. Backend und Worker
   sollen auf einen gesunden Redis-Service warten. `depends_on` ohne Health-Bedingung
   legt nur eine Startreihenfolge fest, keine Bereitschaft. Verbindungsfehler nach dem Start
   verschwinden dadurch nicht automatisch.
2. **Bild-Volume:** Mountet dasselbe benannte Volume bei Backend und Worker unter demselben
   `UPLOAD_FOLDER`. Stellt sicher, dass das Verzeichnis existiert und beschreibbar ist.
   Beim lokalen Start aus `backend/` kann dafür `mkdir -p images` nötig sein.
3. **Redis-Persistenz:** Aktiviert für die Übung AOF (`--appendonly yes`) und bindet ein
   separates Redis-Datenvolume an `/data`. Erklärt, weshalb ein Bild-Volume die Redis-Daten
   nicht schützt und ein Redis-Volume keine JPEG-Dateien des Workers enthält.
4. Verwendet `docker compose down` für den Neustartversuch **ohne `-v`**. Die Option `-v`
   würde die benannten Volumes löschen und damit gerade den Persistenztest verändern.

**Abgabe:** Eure ergänzte Compose-Datei und gegebenenfalls eure Dockerfiles, eine kleine
Architekturskizze und kurze Antworten auf die Fragen in der Tabelle. Fügt einen Nachweis
für die Tests unten bei (z. B. Terminalausgabe mit Job-/Bild-ID und erfolgreichen Requests).
Keine Provider-Keys in Screenshots oder Abgabetext kopieren.


### Schritt D – Abnahmetest und Experimente

Führt die Befehle aus dem Verzeichnis `docker-lab/` aus. Falls ihr andere Servicenamen
gewählt habt, passt sie entsprechend an.

1. **Start:** `docker compose up --build -d`; mit `docker compose ps` den Zustand prüfen.
   Redis muss gesund sein, API und Worker müssen laufen.
2. **Happy Path:** Über Streamlit v3 oder `POST /v3/images` einen Auftrag absenden.
   Die erhaltene `img_id` mit `GET /images/{img_id}` pollen. Erwartet zunächst ggf. 202,
   anschließend 200 mit JPEG. Beobachtet `docker compose logs -f worker`.
3. **Worker-Pause:** `docker compose stop worker`; einen weiteren Auftrag absenden.
   Er bleibt in der Queue, das Bild fehlt. Mit `docker compose start worker` fortsetzen:
   Der vorhandene Auftrag soll ohne erneutes Absenden abgearbeitet werden.
4. **Persistenz:** Stoppt den Worker, erzeugt genau einen wartenden Job und notiert die ID.
   Fahrt den Stack mit `docker compose down` herunter. Startet ihn erneut ohne neue Builds;
   AOF und das Redis-Volume sollen den Job erhalten. Prüft außerdem, ob ein bereits
   generiertes Bild über das Bild-Volume weiter vorhanden ist.
5. **Skalierung (optional):** `docker compose up -d --scale worker=2`.
   Reicht mehrere Jobs ein und beobachtet die Worker-Logs. Vergebt keine feste
   `container_name` für den Worker, wenn ihr ihn skalieren wollt.

**Erfolgskriterien:** Ein Request landet in der vorhandenen `default`-Queue; ein separater
Worker verarbeitet ihn; das Backend kann das Ergebnis aus dem gemeinsamen Volume liefern;
ein pausierter Worker verliert wartende Jobs nicht; Redis-Persistenz und Bild-Persistenz
werden getrennt erklärt und getestet.

**Bonus / ehrliche Grenze des vorhandenen Labs:** Der aktuelle Bild-Polling-Endpunkt
unterscheidet einen noch laufenden Job nicht von einer unbekannten ID oder einem fehlgeschlagenen
Job. Bei einem Providerfehler kann das Frontend deshalb endlos weiterpollen.
Schlagt vor, wie ihr eine Zuordnung von `img_id` zu RQ-Job-ID, Fehlerstatus und Polling-Timeout
ergänzen würdet. Das ist eine Erweiterung, keine Voraussetzung für die Basisaufgabe.


## 10 · Quellen und Vertiefung

Die folgenden offiziellen Dokumentationen erklären die verwendeten Konzepte. Die Demo
bleibt bewusst bei den Versionen in `requirements.txt`; Beispiele neuerer RQ-Versionen
sind nicht immer unverändert übertragbar.

- [Redis: Datentypen](https://redis.io/docs/latest/develop/data-types/)
- [Redis: INCR und Atomarität](https://redis.io/docs/latest/commands/incr/)
- [Redis: TTL](https://redis.io/docs/latest/commands/ttl/)
- [Redis: BLPOP und die Grenzen einer einfachen List-Queue](https://redis.io/docs/latest/commands/blpop/)
- [Redis: Persistenz mit RDB und AOF](https://redis.io/docs/latest/operate/oss_and_stack/management/persistence/)
- [RQ: Queues, Jobs und Anforderungen an Funktionen](https://python-rq.org/docs/)
- [RQ: Verbindungen und Decodierung](https://python-rq.org/docs/connections/)
- [RQ: Worker und SimpleWorker](https://python-rq.org/docs/workers/)
- [FastAPI: Background Tasks](https://fastapi.tiangolo.com/tutorial/background-tasks/)
- [Docker Compose: Netzwerk](https://docs.docker.com/compose/how-tos/networking/)
- [Docker Compose: Startreihenfolge und Bereitschaft](https://docs.docker.com/compose/how-tos/startup-order/)
- [Docker Compose: Volumes](https://docs.docker.com/reference/compose-file/volumes/)

Die SVG-Grafiken unter `figures/` gehören zu diesem Notebook und können auch direkt
für eine Vorführung geöffnet werden.
